# ResearchGPT
RAG pipeline for question answering over research papers with citations.

In [1]:
!pip install -q langchain langchain-community langchain-huggingface sentence-transformers faiss-cpu pymupdf groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 80.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.7/25.7 MB 69.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 54.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [2]:
import os
from getpass import getpass

os.environ['GROQ_API_KEY'] = getpass('Groq API key: ')

Groq API key: ··········


In [3]:
from google.colab import files
uploaded = files.upload()
pdf_paths = list(uploaded.keys())

Saving research paper.pdf to research paper.pdf


In [4]:
import fitz
from dataclasses import dataclass

@dataclass
class PageContent:
    doc_name: str
    page_number: int
    text: str

def extract_pages(pdf_path):
    doc_name = pdf_path.split('/')[-1]
    pages = []
    with fitz.open(pdf_path) as doc:
        for i, page in enumerate(doc):
            text = page.get_text('text').strip()
            if text:
                pages.append(PageContent(doc_name, i + 1, text))
    return pages

all_pages = []
for p in pdf_paths:
    all_pages.extend(extract_pages(p))

len(all_pages)

11

In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150,
    separators=['\n\n', '\n', '. ', ' ', '']
)

documents = []
for page in all_pages:
    for piece in splitter.split_text(page.text):
        documents.append(Document(
            page_content=piece,
            metadata={'doc_name': page.doc_name, 'page_number': page.page_number}
        ))

len(documents)

97

In [8]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2',
    encode_kwargs={'normalize_embeddings': True}
)

vector_store = FAISS.from_documents(documents, embeddings)
vector_store.index.ntotal

/tmp/ipykernel_5835/1839299023.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

97

In [9]:
from groq import Groq

client = Groq(api_key=os.environ['GROQ_API_KEY'])

PROMPT_TEMPLATE = '''You are ResearchGPT, an assistant that answers questions strictly using the provided research paper excerpts.
Only use information found in the CONTEXT. If the answer isn't there, say so honestly. Be concise.

CONTEXT:
{context}

QUESTION: {question}

ANSWER:'''

def ask(question, k=4):
    results = vector_store.similarity_search_with_relevance_scores(question, k=k)
    context = '\n\n'.join(
        f"[Source {i+1} | {doc.metadata['doc_name']} | Page {doc.metadata['page_number']}]\n{doc.page_content}"
        for i, (doc, score) in enumerate(results)
    )
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    response = client.chat.completions.create(
        model='llama-3.1-8b-instant',
        messages=[{'role': 'user', 'content': prompt}],
        temperature=0.2,
        max_tokens=800,
    )
    answer = response.choices[0].message.content.strip()

    print(answer)
    print()
    for doc, score in results:
        print(f"{doc.metadata['doc_name']} (page {doc.metadata['page_number']}, relevance {score:.3f})")
    return answer, results

In [10]:
_ = ask('What problem does this paper try to solve, and what method do they propose?')

/tmp/ipykernel_5835/3600050222.py:16: UserWarning: Relevance scores must be between 0 and 1, got [(Document(id='44364391-b206-4975-83f4-7b4b326ac445', metadata={'doc_name': 'research paper.pdf', 'page_number': 4}, page_content='is to minimize the log-probability that the discriminator correctly\ndistinguishes samples from G. Because the sampling of s is discrete,\nwe follow [21, 25, 33] to compute the gradient of V (G, D) with\nrespect to θG by policy gradient:\n∇θG V (G, D) = ∇θG\nV\nÕ\nc=1\nEs∼G(·|vc )[log(1 −D(s))]\n=\nV\nÕ\nc=1\nEs∼G(·|vc )[∇θG logG(s |vc) log(1 −D(s))].\n(3)\n4.3\nA Naive Implementation of D and G\nA naive implementation of the discriminator and generator is based\non sigmoid and softmax functions, respectively.\nFor the discriminator D, intuitively we can define it as the multi-\nplication of the sigmoid function of the inner product of every two\nvertices in the input vertex subset s:\nD(s) =\nÖ\n(u,v)∈s,u,v\nσ(d⊤\nu · dv ),\n(4)\nwhere du, dv ∈Rk are the k-dime

Based on the provided research paper excerpts, the problem this paper tries to solve is the dense overlapping problem in community detection, and the motif generation and discrimination.

The proposed method is called CommunityGAN, which is a framework that uses a generator and a discriminator to optimize the motif-level representation and solve the dense overlapping problem.

research paper.pdf (page 4, relevance -0.005)
research paper.pdf (page 4, relevance -0.023)
research paper.pdf (page 6, relevance -0.041)
research paper.pdf (page 10, relevance -0.042)
